  # **PROVEEDORES**





In [4]:
pip install faker

Note: you may need to restart the kernel to use updated packages.


c:\prueba\.venv\Scripts\python.exe: No module named pip


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

from pyspark.sql.types import IntegerType

# Obtener la sesión de Spark existente
spark = SparkSession.builder.getOrCreate()


In [6]:

import json
config_proveedor = {
  "seed": 4321,
  "volumen": {
    "proveedores": 800
  },

  "paises": ["colombia", "chile", "china", "usa", "suiza"],

     "tiempo_repo": {
        "minimo": 10,
        "maximo": 25
    },

  "calificacion_calidad": {
        "minimo": 1,
        "maximo": 5
    }


}

with open("config_proveedor.json", "w", encoding="utf-8") as f:
    json.dump(config_proveedor, f, indent=4, ensure_ascii=False)

print("Archivo config.json creado correctamente.")

Archivo config.json creado correctamente.


In [12]:
from faker import Faker


with open("config_proveedor.json", "r", encoding="utf-8") as f:
    config = json.load(f)


fake = Faker("es_CO")
from faker.providers import DynamicProvider

pais = DynamicProvider(
     provider_name="pais",
     elements=config["paises"],
)

fake = Faker()
fake.seed_instance(4321)

# then add new provider to faker instance
fake.add_provider(pais)


datos_proveedores = []

for i in range(config["volumen"]["proveedores"]):
  id_proveedor = i + 1
  datos_proveedores.append({
        "id_proveedor":id_proveedor,
        "razon_social": fake.company(),
        "tiempo_repo_dias": fake.random_int(
            min=config["tiempo_repo"]["minimo"],
            max=config["tiempo_repo"]["maximo"] ),
        "calificacion_calidad": fake.pyfloat(
            min_value=config["calificacion_calidad"]["minimo"],
            max_value=config["calificacion_calidad"]["maximo"],right_digits=1 , positive=True),
        "pais":fake.pais(),
        "activo":fake.pybool()

    })



In [16]:
import os

# Esto crea las carpetas si no existen de forma silenciosa
os.makedirs("Files/datos_sinteticos", exist_ok=True)

# Convertimos el DataFrame de PySpark a un DataFrame de Pandas
df_pandas = df_faker.toPandas()

# Lo guardamos directamente como archivo CSV
df_pandas.to_csv("Files/datos_sinteticos/proveedores.csv", index=False, encoding="utf-8")

c:\prueba\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\prueba\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


In [17]:
# Convertimos el DataFrame de PySpark a un DataFrame de Pandas
df_pandas = df_faker.toPandas()

# Lo guardamos directamente como archivo JSON
df_pandas.to_json(
    "Files/datos_sinteticos/proveedores.json", 
    orient="records", 
    force_ascii=False, 
    indent=4 # Esto lo hace fácil de leer para el ojo humano
)

c:\prueba\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\prueba\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


In [18]:
import pandas as pd


# **ARTICULOS**

In [19]:
config_articulo = {
  "seed": 4321,
  "volumen": {
    "articulos": 5000
  },

  "categorias": {
    "Alimentos y bebidas": {
      "Lacteos": ["Leche Entera", "Yogurt", "Queso", "Mantequilla"],
      "Panaderia": ["Pan Tajado", "Pan Artesanal", "Tortillas"],
      "Bebidas": ["Gaseosas", "Jugos", "Agua Embotellada", "Cervezas"],
      "Abarrotes": ["Arroz", "Aceite", "Pasta", "Granos"]
    },
    "Cuidado personal e higiene": {
      "Higiene Bucal": ["Cepillos Dentales", "Cremas Dentales", "Enjuague Bucal"],
      "Cuidado Capilar": ["Shampoo", "Acondicionador", "Tratamientos"],
      "Higiene Corporal": ["Jabones", "Desodorantes", "Papel Higienico"]
    },
    "Hogar y limpieza": {
      "Limpieza General": ["Detergentes", "Desinfectantes", "Limpiavidrios"],
      "Lavanderia": ["Suavizantes", "Blanqueadores", "Jabon en Polvo"],
      "Utensilios Hogar": ["Bolsas de Basura", "Servilletas", "Papel Aluminio"]
    },
    "Electronica y tecnologia": {
      "Accesorios": ["Cables", "Cargadores", "Audifonos"],
      "Pequenos Electrodomesticos": ["Licuadoras", "Planchas", "Cafeteras"],
      "Computo": ["Mouse", "Teclados", "Memorias USB"]
    },
    "Ropa y calzado basico": {
      "Ropa Interior": ["Bóxers", "Calcetines", "Medias"],
      "Ropa Casual": ["Camisetas", "Pantalones", "Sudaderas"],
      "Calzado": ["Tenis", "Sandalias", "Botas"]
    },
    "Bebes y maternidad": {
      "Panales e Higiene": ["Panales", "Toallitas Humedas", "Cremas Antipanalitis"],
      "Alimentacion Infantil": ["Formula Lactea", "Papillas", "Cereales Infantiles"],
      "Maternidad": ["Fajas Postparto", "Protectores Lactancia"]
    }

},


  "precio_lista": {
    "minimo": 1000,
    "maximo": 150000
  },
  "peso_kg": {
    "minimo": 0.02,
    "maximo": 25.0
  },
  "unid_medida": ["UND", "KG", "LT", "PQ", "CJ"],
  "activo": {
    "pct_activos": 0.90
  },
  "fec_alta": {
    "inicio": "2015-01-01",
    "fin": "2026-07-25"
  }
}

with open("config_articulo.json", "w", encoding="utf-8") as f:
    json.dump(config_articulo, f, indent=4, ensure_ascii=False)

print("Archivo config.json creado correctamente.")

Archivo config.json creado correctamente.


In [20]:
import json
from datetime import datetime

with open("config_articulo.json", "r", encoding="utf-8") as f:
    config = json.load(f)


fecha_ini = datetime.strptime(
    config["fec_alta"]["inicio"], "%Y-%m-%d"
).date()

fecha_fin = datetime.strptime(
    config["fec_alta"]["fin"], "%Y-%m-%d"
).date()

In [21]:
from faker import Faker
from faker.providers import DynamicProvider
import json
import random
import datetime

# Leer configuracion (ojo: el archivo se llama config_articulo.json, no config.json)
with open("config_articulo.json", "r", encoding="utf-8") as f:
    config = json.load(f)

fake = Faker("es_CO")
fake.seed_instance(config["seed"])

def generar_categoria(fake, categorias: dict):
    macro = fake.random_element(elements=list(categorias.keys()))
    categoria = fake.random_element(elements=list(categorias[macro].keys()))
    subcategoria = fake.random_element(elements=categorias[macro][categoria])
    return macro, categoria, subcategoria


uni_medida = DynamicProvider(
    provider_name="uni_medida",
      elements=config["unid_medida"]

)

fake.add_provider(uni_medida)



macro, categoria, subcategoria = generar_categoria(fake, config["categorias"])

datos_articulos = []

for i in range(config["volumen"]["articulos"]):



        datos_articulos.append({
        "art_id": i + 1,
        "cod_barra": fake.ean13(),

         "id_categ_n1": macro,
        "id_categ_n2": categoria,
        "id_categ_n3": subcategoria,
        "id_proveedor":id_proveedor ,
        "precio_lista": fake.pyfloat(
            min_value=config["precio_lista"]["minimo"],
            max_value=config["precio_lista"]["maximo"],
            right_digits=0, positive=True
        ),
        "peso_kg": fake.pyfloat(
            min_value=config["peso_kg"]["minimo"],
            max_value=config["peso_kg"]["maximo"],
            right_digits=2, positive=True
        ),
        "unid_medida": fake.uni_medida(),
        "activo": fake.pybool(),
        "fec_alta": fake.date_between(start_date=fecha_ini, end_date=fecha_fin),
    })



In [25]:


df_pandas = pd.DataFrame(datos_articulos)

# Guardamos en CSV (o cámbialo a .to_json si prefieres)
df_pandas.to_csv("Files/datos_sinteticos/articulos.csv", index=False, encoding="utf-8")





df_pandas.to_csv("Files/datos_sinteticos/articulo.csv",
 index=False, 
 encoding="utf-8")

# **TIENDA**

In [26]:
import json

config_tienda= {
    "seed": 2765,

    "volumen": {
        "tiendas": 150
    },

    "fechas": {
        "inicio": "2020-01-01",
        "fin": "2025-12-31"
    },

    "tiendas": {
        "tipos": [
            "hipermercados",
            "supermercado",
            "tiendas de conveniencia"
        ]
    },

    "paises": {
        "colombia": ["Bogotá", "Medellín", "Cali", "Barranquilla"],
        "chile": ["Santiago", "Valparaíso", "Concepción"],
        "peru": ["Lima", "Arequipa", "Cusco"],
        "ecuador": ["Quito", "Guayaquil", "Cuenca"]
    },

    "metros_cuadrados": {
        "minimo": 80,
        "maximo": 5000
    }
}

with open("config_tienda.json", "w", encoding="utf-8") as f:
    json.dump(config_tienda, f, indent=4, ensure_ascii=False)

print("Archivo config.json creado correctamente.")

Archivo config.json creado correctamente.


In [27]:
import json
from datetime import datetime

with open("config_tienda.json", "r", encoding="utf-8") as f:
    config = json.load(f)

fecha_inicio = datetime.strptime(
    config["fechas"]["inicio"], "%Y-%m-%d"
).date()

fecha_fin = datetime.strptime(
    config["fechas"]["fin"], "%Y-%m-%d"
).date()

In [28]:
from faker import Faker
from faker.providers import DynamicProvider
import json
import random

# Leer configuración
with open("config_tienda.json", "r", encoding="utf-8") as f:
    config = json.load(f)

fake = Faker("es_CO")
fake.seed_instance(config["seed"])

tipo_tiendas = DynamicProvider(
    provider_name="tienda",
    elements=config["tiendas"]["tipos"]
)

pais_retail = DynamicProvider(
    provider_name="paises",
    elements=list(config["paises"].keys())
)

fake.add_provider(tipo_tiendas)
fake.add_provider(pais_retail)

ciudades_por_pais = config["paises"]



datos_tienda = []

for i in range(config["volumen"]["tiendas"]):

    pais = fake.paises()
    id_tienda = i + 1

    datos_tienda.append({
        "id_tienda": id_tienda,
        "tipo_tienda": fake.tienda(),
        "nom_tienda": fake.company_suffix() if random.random() > 0.05 else None,
        "pais": pais,
        "ciudad": fake.random_element(ciudades_por_pais[pais]),
        "fecha_apertura": fake.date_between(
            start_date=fecha_inicio,
            end_date=fecha_fin
        ),
        "metros_cuadrados": fake.random_int(
            min=config["metros_cuadrados"]["minimo"],
            max=config["metros_cuadrados"]["maximo"]
        ),
        "activo": fake.pybool()
    })



In [29]:
df_pandas = pd.DataFrame(datos_tienda)


df_pandas.to_csv("Files/datos_sinteticos/tienda.csv", index=False, encoding="utf-8")




df_pandas.to_json(
    "Files/datos_sinteticos/tienda.json", 
    orient="records", 
    force_ascii=False, 
    indent=4 
)

# **MIEMBROS**

In [30]:
import json

config_miembros = {

    "seed": 4827,

    "volumen": {
        "miembros": 50000
    },

    "fechas_registro":{

            "inicio": "2020-01-01",
            "fin": "2023-12-31"
        },


        "fechas_ultima_compra": {

                "inicio": "2024-01-01",
                "fin": "2025-12-31"

        }
    ,





    "ciudades":

       [ "Bogota", "Medellin", "Cali", "Barranquilla", "Cartagena", "Bucaramanga", "Pereira", "Manizales", "Cucuta", "Santa Marta",
            "Buenos Aires", "Cordoba", "Rosario", "Mendoza", "La Plata", "Mar del Plata", "Salta", "Tucuman",

           "Santiago de Chile", "Valparaiso", "Concepcion", "Antofagasta", "La Serena", "Temuco", "Vina del Mar", "Rancagua", "Puerto Montt",
           "Quito", "Guayaquil", "Cuenca", "Ambato", "Manta", "Machala", "Loja", "Riobamba",
         "Ciudad de Mexico", "Guadalajara", "Monterrey", "Puebla", "Tijuana", "Leon", "Queretaro", "Merida", "Cancun"

        ],


"generos": [
        "Femenino",
        "Masculino",
        "Otro"
    ],

    "rangos_edad": [
        "18-20",
        "21-25",
        "26-30",
        "31-35",
        "36-40",
        "41-50",
        "51-60",
        "61+"
    ],

    "canales_preferidos": [
        "tienda fisica",
        "pagina web",
        "app"
    ]
}

with open("config_miembros.json", "w", encoding="utf-8") as f:
    json.dump(config_miembros, f, indent=4, ensure_ascii=False)

print("Archivo config_miembros.json creado correctamente.")

Archivo config_miembros.json creado correctamente.


In [31]:


import json
from datetime import datetime
with open("config_miembros.json", "r", encoding="utf-8") as f:
    config = json.load(f)

fecha_registro_ini= datetime.strptime(
    config["fechas_registro"]["inicio"], "%Y-%m-%d" ).date()

fecha_registro_fin = datetime.strptime(
    config["fechas_registro"]["fin"], "%Y-%m-%d"
).date()



fecha_com_ini= datetime.strptime(
    config["fechas_ultima_compra"]["inicio"], "%Y-%m-%d" ).date()

fecha_com_fin = datetime.strptime(
    config["fechas_ultima_compra"]["fin"], "%Y-%m-%d"
).date()

In [32]:
from faker import Faker
from faker.providers import DynamicProvider
import json

import json

with open("config_miembros.json", "r", encoding="utf-8") as f:
    config = json.load(f)

fake = Faker("es_CO")
fake.seed_instance(config["seed"])

edades = DynamicProvider(
    provider_name="edad",
    elements=config["rangos_edad"]
)

generos= DynamicProvider(
    provider_name="genero",
       elements=config["generos"]

)
canales_preferidos= DynamicProvider(
    provider_name="canal_pref",
       elements=config["canales_preferidos"]

)

ciudades= DynamicProvider(
    provider_name="ciudad",
       elements=config["ciudades"]

)

fake.add_provider(canales_preferidos)
fake.add_provider(generos)
fake.add_provider(edades)
fake.add_provider(ciudades)

ciudades_por_pais = config["ciudades"]

datos_miembros = []

for i in range(config["volumen"]["miembros"]):

      id_cliente = i + 1
      datos_miembros.append({
        "id_miembro": id_cliente,
        "canal_preferido": fake.canal_pref(),
        "genero": fake.genero(),
        "ciudad": fake.ciudad(),
         "fecha_registro": fake.date_between( start_date=fecha_registro_ini, end_date=fecha_registro_fin ),
         "fecha_ultima_compra": fake.date_between( start_date=fecha_com_ini, end_date=fecha_com_fin ),
        "edad": fake.edad(),
        "activo": fake.pybool()
    })



In [33]:
df_pandas = pd.DataFrame(datos_miembros)


df_pandas.to_csv("Files/datos_sinteticos/miembros.csv", index=False, encoding="utf-8")




df_pandas.to_json(
    "Files/datos_sinteticos/miembros.json", 
    orient="records", 
    force_ascii=False, 
    indent=4 
)

# **VENTAS**

In [34]:
config_ventas ={
    "seed": 9315,

    "volumen": {
        "transacciones": 1000000
    },

    "fechas_transaccion": {
        "inicio": "2023-01-01",
        "fin": "2025-12-31"
    },

    "cantidad_vendida": {
        "minimo": 1,
        "maximo": 25
    },

    "precio_unitario": {
        "minimo": 5000,
        "maximo": 500000
    },

    "descuento": {
        "valores": [
            0,
            5,
            10,
            15,
            20,
            30,
            40,
            50
        ]
    },

    "tipos_pago": [
        "Efectivo",
        "Tarjeta Debito",
        "Tarjeta Credito",
        "PayPal"

    ],

    "canales_venta": [
        "tienda fisica",
        "pagina web",
        "app"
    ]

,

    "horarios": {
    "madrugada": {
        "inicio": 0,
        "fin": 7,
        "probabilidad": 5
    },
    "mañana": {
        "inicio": 8,
        "fin": 11,
        "probabilidad": 20
    },
    "mediodia": {
        "inicio": 12,
        "fin": 15,
        "probabilidad": 35
    },
    "tarde": {
        "inicio": 16,
        "fin": 19,
        "probabilidad": 30
    },
    "noche": {
        "inicio": 20,
        "fin": 23,
        "probabilidad": 10
    },
}
}

with open("config_ventas.json", "w", encoding="utf-8") as f:
    json.dump(config_ventas, f, indent=4, ensure_ascii=False)

In [35]:
import json
from datetime import datetime
import json
with open("config_ventas.json", "r", encoding="utf-8") as f:
    config = json.load(f)

fecha_trans_ini= datetime.strptime(
    config["fechas_transaccion"]["inicio"], "%Y-%m-%d" ).date()

fecha_trans_fin = datetime.strptime(
    config["fechas_transaccion"]["fin"], "%Y-%m-%d"
).date()



In [36]:
from faker import Faker
from faker.providers import DynamicProvider
import json

import json
with open("config_ventas.json", "r", encoding="utf-8") as f:
    config = json.load(f)


fake = Faker("es_CO")
fake.seed_instance(config["seed"])

def generar_hora(config):

    horarios = list(config["horarios"].values())

    bloque = random.choices(
        horarios,
        weights=[h["probabilidad"] for h in horarios],
        k=1
    )[0]

    hora = random.randint(
        bloque["inicio"],
        bloque["fin"]
    )

    minuto = random.randint(0, 59)


    return f"{hora:02d}:{minuto:02d}"

horarios = config["horarios"]

tipo_pago = DynamicProvider(
    provider_name="tipo_pago",
    elements=config["tipos_pago"]
)

descuentos = DynamicProvider(
    provider_name="descuento",
       elements=config["descuento"]["valores"]

)
canales_ventas = DynamicProvider(
    provider_name="canales_ventas",
       elements=config["canales_venta"]

)




fake.add_provider(canales_ventas)
fake.add_provider(descuentos)
fake.add_provider(tipo_pago)




datos_venta = []

for i in range(config["volumen"]["transacciones"]):


  fecha_trans = fake.date_between( start_date=fecha_trans_ini, end_date=fecha_trans_fin )
  cantidad_vendida = fake.random_int(
        min=config["cantidad_vendida"]["minimo"],
        max=config["cantidad_vendida"]["maximo"]  )

datos_venta.append({
        "id_transacion": i + 1,
        "fecha_trans":fecha_trans,
        "cantidad_vendida":cantidad_vendida,
        "id_miembro":id_cliente,
        "id_tienda":id_tienda,

        "precio_unitario_venta":fake.random_int(
            min=config["precio_unitario"]["minimo"],
            max=config["precio_unitario"]["maximo"]  ),
        "descuento":fake.descuento(),
        "total_venta": fake.random_int(
            min=config["precio_unitario"]["minimo"],
            max=config["precio_unitario"]["maximo"]  ),
        "tipo_pago":fake.tipo_pago(),
         "hora_trans": generar_hora(config),
        "canal_venta":fake.canales_ventas()


    })



In [37]:
df_pandas = pd.DataFrame(datos_venta)


df_pandas.to_csv("Files/datos_sinteticos/venta.csv", index=False, encoding="utf-8")




df_pandas.to_json(
    "Files/datos_sinteticos/venta.json", 
    orient="records", 
    force_ascii=False, 
    indent=4 
)

# **STOK DIARIO**

In [38]:
config_stock = {
  "seed": 4576,
  "volumen": {
    "snapshots": 750000
  },

    "fecha_snapshot": {
        "inicio": "2023-01-01",
        "fin": "2025-12-31"
    },
  "stock_minimo_config": {
    "minimo": 10,
    "maximo": 50
  },
  "multiplicador_maximo": {
    "minimo": 3,
    "maximo": 8
  },
  "pct_transito_max": 0.30,
  "pct_sobrestock_max": 1.20
}
with open("config_stock.json", "w", encoding="utf-8") as f:
    json.dump(config_stock, f, indent=4, ensure_ascii=False)

In [39]:
import json
from datetime import datetime
import json
with open("config_stock.json", "r", encoding="utf-8") as f:
    config = json.load(f)


fecha_stock_ini= datetime.strptime(
    config["fecha_snapshot"]["inicio"], "%Y-%m-%d" ).date()

fecha_stock_fin = datetime.strptime(
    config["fecha_snapshot"]["fin"], "%Y-%m-%d"
).date()


In [40]:
from faker import Faker
from faker.providers import DynamicProvider
import json

import json
with open("config_stock.json", "r", encoding="utf-8") as f:
    config = json.load(f)


fake = Faker("es_CO")
fake.seed_instance(config["seed"])

datos_stock = []

for i in range(config["volumen"]["snapshots"]):

    stock_minimo_config = fake.random_int(
        min=config["stock_minimo_config"]["minimo"],
        max=config["stock_minimo_config"]["maximo"]
    )

    stock_maximo_config = stock_minimo_config * fake.random_int(
        min=config["multiplicador_maximo"]["minimo"],
        max=config["multiplicador_maximo"]["maximo"]
    )

    stock_fisico = fake.random_int(
        min=0,
        max=int(stock_maximo_config * config["pct_sobrestock_max"])
    )

    stock_transito = fake.random_int(
        min=0,
        max=int(stock_maximo_config * config["pct_transito_max"])
    )


    stock_reservado = fake.random_int(min=0, max=stock_fisico) if stock_fisico > 0 else 0

    datos_stock.append({
        "id_snapshot": i + 1,
        "stock_minimo_config": stock_minimo_config,
        "stock_maximo_config": stock_maximo_config,
        "stock_fisico": stock_fisico,
        "stock_transito": stock_transito,
        "stock_reservado": stock_reservado,
        "fecha_trans":fake.date_between( start_date=fecha_stock_ini, end_date=fecha_stock_fin ),
    })


In [41]:
df_pandas = pd.DataFrame(datos_stock)


df_pandas.to_csv("Files/datos_sinteticos/stock.csv", index=False, encoding="utf-8")




df_pandas.to_json(
    "Files/datos_sinteticos/stock.json", 
    orient="records", 
    force_ascii=False, 
    indent=4 
)

# **DEVOLUCIONES**

In [42]:

config_devolucion= {
  "seed": 42,
  "volumen": {
    "devoluciones": 50000
  },
   "fecha_snapshot": {
        "inicio": "2023-01-01",
        "fin": "2025-12-31"
    },
  "devoluciones": {
    "pct_transacciones": 0.08 },

    "motivos": ["DEFECTUOSO",
                "TALLA",
                "NO_CUMPLE_EXPECT",
                "ARREPENTIMIENTO"],

    "canales": ["Tienda fisica",
                "Recogida a domicilio",
                "Correo"],
    "estados": ["Pendiente",
                "Aprobada",
                "Rechazada",
                "Reembolsada"]

}

with open("config_devolucion.json", "w", encoding="utf-8") as f:
    json.dump(config_devolucion, f, indent=4, ensure_ascii=False)

In [43]:
from faker import Faker
from faker.providers import DynamicProvider
import json
import datetime

import json
with open("config_devolucion.json", "r", encoding="utf-8") as f:
    config = json.load(f)


fake = Faker("es_CO")
fake.seed_instance(config["seed"])



estado = DynamicProvider(
    provider_name="estados",
    elements=config["estados"]
)

canales = DynamicProvider(
    provider_name="canales",
    elements=config["canales"]

)


motivos = DynamicProvider(
    provider_name="motivos",
    elements=config["motivos"]

)

fake.add_provider(motivos)
fake.add_provider(estado)
fake.add_provider(canales)

datos_devolucion = []

for i in range(config["volumen"]["devoluciones"]):


                datos_devolucion .append({
                         "id_devolucion": i + 1,
                         "qty_devuelta":fake.random_int(min=1, max=cantidad_vendida),
                         "motivo":fake.motivos(),
                         "estado":fake.estados(),
                         "canal":fake.canales(),

                          "fecha_trans":fake.date_between(
                            start_date=fecha_trans,
                             end_date=fecha_trans + datetime.timedelta(days=30)
                                 )
    })


In [44]:
df_pandas = pd.DataFrame(datos_devolucion)


df_pandas.to_csv("Files/datos_sinteticos/devolucion.csv", index=False, encoding="utf-8")




df_pandas.to_json(
    "Files/datos_sinteticos/devolucion.json", 
    orient="records", 
    force_ascii=False, 
    indent=4 
)